# E-Commerce Sales & Customer Analytics
### Step 2 & 3: Data Cleaning and Exploratory Data Analysis (EDA)

This notebook covers the comprehensive data cleaning and exploratory analysis of our raw e-commerce transaction dataset (100,000+ records).

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Load Datasets

In [ ]:
customers = pd.read_csv('../Dataset/raw/customers.csv')
products = pd.read_csv('../Dataset/raw/products.csv')
orders = pd.read_csv('../Dataset/raw/orders.csv')
order_items = pd.read_csv('../Dataset/raw/order_items.csv')
payments = pd.read_csv('../Dataset/raw/payments.csv')
reviews = pd.read_csv('../Dataset/raw/reviews.csv')
returns = pd.read_csv('../Dataset/raw/returns.csv')

print(f"Raw Orders shape: {orders.shape}")
print(f"Raw Order Items shape: {order_items.shape}")

## 2. Null Value Analysis & Deduplication

In [ ]:
# Duplicate Check
print(f"Duplicate Customer Records: {customers.duplicated(subset=['customer_id']).sum()}")
customers.drop_duplicates(subset=['customer_id'], inplace=True)
orders.drop_duplicates(subset=['order_id'], inplace=True)

# Impute missing zip code prefix
customers['customer_zip_code_prefix'].fillna('00000', inplace=True)

# Order missing dates imputation
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Impute delivery date for delivered status with default (7 days after purchase)
mask = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isna())
orders.loc[mask, 'order_delivered_customer_date'] = orders.loc[mask, 'order_purchase_timestamp'] + pd.Timedelta(days=7)

## 3. Outlier Correction

In [ ]:
# Inspect weight anomalies
print(f"Negative weights count: {(products['product_weight_g'] < 0).sum()}")
products['product_weight_g'] = products['product_weight_g'].abs()
products['product_base_price'] = products['product_base_price'].abs()

# Impute pricing outliers ($99,999 cases)
category_medians = products[products['product_base_price'] < 50000].groupby('product_category_name')['product_base_price'].median()
products.loc[products['product_base_price'] > 50000, 'product_base_price'] = products.loc[products['product_base_price'] > 50000, 'product_category_name'].map(category_medians)

## 4. Feature Engineering

In [ ]:
# Delivery Days
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
# Delivery delay relative to estimate
orders['delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_delayed'] = orders['delay_days'] > 0

# Total spend per item line
order_items['item_value'] = order_items['price'] * order_items['quantity']
order_totals = order_items.groupby('order_id')['item_value'].sum().reset_name = 'order_value'

## 5. Visual EDA

In [ ]:
# Plot Monthly Sales Trends
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
order_items_merged = order_items.merge(orders, on='order_id')
monthly_sales = order_items_merged.groupby('year_month')['item_value'].sum()

monthly_sales.plot(kind='line', marker='o', color='#38bdf8', linewidth=2.5)
plt.title('Monthly Sales Revenue Growth (2024-2026)')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.show()

## 6. Advanced Customer Analytics: RFM Analysis

In [ ]:
# Reference max date
max_date = orders['order_purchase_timestamp'].max()
user_rfm = order_items_merged.groupby('customer_id').agg({
    'order_purchase_timestamp': lambda x: (max_date - x.max()).days,
    'order_id': 'nunique',
    'item_value': 'sum'
}).rename(columns={
    'order_purchase_timestamp': 'recency',
    'order_id': 'frequency',
    'item_value': 'monetary'
})

print(user_rfm.head())